# AnalogML — Walkthrough Notebook

This notebook walks through the full AnalogML workflow:
1. Generate synthetic OTA dataset
2. Parse netlist → circuit graph
3. Train physics-backed model
4. Predict specs without simulation
5. Inverse design (size recommendation)
6. Technology transfer
7. Symbolic equation extraction

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

from analogml.parsers import SpiceParser
from analogml.core    import CircuitGraph, FeatureExtractor
from analogml.models  import AnalogMLModel, TechTransferAgent
from analogml.data    import SyntheticCircuitDataset

## 1. Synthetic OTA Dataset (180nm, 5T topology)

In [ ]:
synth = SyntheticCircuitDataset(seed=42)
X, Y, x_names, y_names = synth.generate('ota_5t', n_samples=300, technology='180nm')
print(f'X: {X.shape}  Y: {Y.shape}')
print('Output specs:', y_names)

fig, axes = plt.subplots(2, 4, figsize=(14, 5))
for i, (name, ax) in enumerate(zip(y_names, axes.flatten())):
    ax.hist(Y[:, i], bins=20, color='#2196F3', alpha=0.8)
    ax.set_title(name, fontsize=9)
plt.suptitle('Distribution of Output Specs', y=1.01)
plt.tight_layout()
plt.show()

## 2. Netlist Parsing & Graph Encoding

In [ ]:
import networkx as nx

netlist_text = '''
* 5T OTA
.subckt ota INP INN OUT VDD VSS VBIAS
M1 net1 INP net3 VSS nmos W=4u L=0.18u
M2 net2 INN net3 VSS nmos W=4u L=0.18u
M3 net1 net1 VDD VDD pmos W=8u L=0.18u
M4 net2 net1 VDD VDD pmos W=8u L=0.18u
M5 net3 VBIAS VSS VSS nmos W=8u L=0.36u
Cc net2 OUT 500f
Cload OUT VSS 2p
.ends
'''

parser  = SpiceParser()
netlist = parser.parse_text(netlist_text, technology='180nm')
graph   = CircuitGraph.from_netlist(netlist)

print(f'Nodes: {list(graph.G.nodes)}')
print(f'Edges: {graph.G.number_of_edges()}')

# Visualize
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
G_simp = nx.Graph(graph.G)
colors = ['#2196F3' if 'nmos' in n else
          '#F44336' if 'pmos' in n else
          '#4CAF50' for n in G_simp.nodes]
pos = nx.spring_layout(G_simp, seed=7)
nx.draw(G_simp, pos, labels={n:n for n in G_simp}, node_color=colors,
        node_size=1200, font_size=8, ax=ax, edge_color='gray')
ax.set_title('OTA Circuit Graph\n(Blue=NMOS, Red=PMOS, Green=Passive)')
plt.show()

## 3. Train Physics-Backed Model

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_te, Y_tr, Y_te = train_test_split(X, Y, test_size=0.2, random_state=0)

model = AnalogMLModel(mode='fast', output_names=y_names)
model.fit(X_tr, Y_tr, X_val=X_te, Y_val=Y_te, epochs=200)
print(f'Training done. Final loss = {model.history["loss"][-1]:.5f}')

## 4. Predict Without Simulation

In [ ]:
Y_pred, Y_std = model.predict_with_uncertainty(X_te)

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i, (name, ax) in enumerate(zip(y_names, axes.flatten())):
    ax.scatter(Y_te[:,i], Y_pred[:,i], alpha=0.5, s=12, c='#2196F3')
    mn, mx = Y_te[:,i].min(), Y_te[:,i].max()
    ax.plot([mn,mx],[mn,mx],'r--',lw=1.5)
    r2 = np.corrcoef(Y_te[:,i], Y_pred[:,i])[0,1]**2
    ax.set_title(f'{name}\nR²={r2:.3f}', fontsize=8)
    ax.set_xlabel('True', fontsize=7)
    ax.set_ylabel('Pred', fontsize=7)
plt.suptitle('Prediction vs Truth — OTA 180nm', y=1.01)
plt.tight_layout()
plt.show()

## 5. Inverse Design — Size Recommendation

In [ ]:
target = {
    'gain_dB'          : 65.0,
    'ugf_MHz'          : 15.0,
    'phase_margin_deg' : 60.0,
    'power_uW'         : 40.0,
}

x_best = model.recommend_sizes(target, X_candidates=X_tr)
y_best = model.predict(x_best.reshape(1,-1))[0]

print('Recommended sizes:')
for n, v in zip(x_names, x_best):
    print(f'  {n:<20} = {v:.3f}')
print()
print('Expected performance:')
for n, v, t in zip(y_names, y_best, [target.get(k) for k in y_names]):
    marker = '✓' if t is None else ('✓' if abs(v-t)/max(abs(t),1e-9)<0.15 else '⚠')
    print(f'  {marker} {n:<30} pred={v:.2f}  target={t}')

## 6. Technology Transfer 180nm → 90nm

In [ ]:
synth_90 = SyntheticCircuitDataset(seed=99)
X_90, Y_90, _, _ = synth_90.generate('ota_5t', n_samples=200, technology='90nm')

agent = TechTransferAgent('180nm', '90nm')
agent.fit(X_tr, X_90[:len(X_tr)])

x_180 = X_te[0]
x_90_est = agent.transfer(x_180)[0]
print('Original 180nm:')
for n, v in zip(x_names, x_180): print(f'  {n:<20} = {v:.3f}')
print()
print('Estimated 90nm sizes:')
for n, v in zip(x_names, x_90_est): print(f'  {n:<20} = {v:.3f}')

rules = agent.tech_scaling_rules()
print()
print('Scaling rules:')
for k, v in rules.items():
    bar = '█' * int(abs(v) * 10)
    print(f'  {k:<25}  ×{v:.3f}  {bar}')